# Estimación Automática del Ángulo de Cobb en Radiografías de Columna Vertebral mediante Detección de Puntos Clave con YOLOv11-Pose

**Materia:** Machine Learning  
**Profesor:** Dr. Luis Carlos Padierna García  
**Alumna:** Farah Rashid Bermúdez Orozco\
**NUA:** 446769 \
**Institución:** Universidad de Guanajuato, División de Ciencias e Ingenierías  \
**Fecha:** Mayo 2026  

---

## Descripción general

La escoliosis es una deformidad de la columna vertebral que afecta al 2–3% de la población mundial. Su diagnóstico depende de la medición manual del **ángulo de Cobb** que es un proceso lento, subjetivo y con variabilidad entre observadores de hasta 10°. El software clínico de referencia en Estados Unidos (EOS imaging con spineEOS) tiene un costo de instalación de aproximadamente 950,000 USD, lo que lo hace inaccesible en países en desarrollo.

Este proyecto propone una solución basada en **YOLOv11-Pose** para estimar automáticamente los tres ángulos de Cobb (PT, MT y TL) a partir de radiografías de columna en proyección anteroposterior (AP). El modelo detecta los puntos clave anatómicos de las 17 vértebras torácicas y lumbares y calcula los ángulos utilizando geometría.

---

## Versiones de librerías utilizadas

| Librería | Versión | 
|---|---|
| PyTorch | 2.5.1+cu121 |
| Ultralytics | 8.4.51 | 
| NumPy | 1.26.4 |
| OpenCV | 4.9.0.80 |

In [11]:
#Librerías a importar
import os
import json
import numpy as np
import matplotlib
matplotlib.use('Agg') #Evita que colapse el kernel
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import cv2
import torch

#Versiones para portabilidad
print(f"Python:  3.10.20  |  NumPy: {np.__version__}  |  "
      f"OpenCV: {cv2.__version__}  |  PyTorch: {torch.__version__}")

#Rutas de las carpetas a utilizar
RUTA_JSON_TRAIN = "./data/annotations/train.json"
RUTA_JSON_TEST = "./data/annotations/test.json"
RUTA_IMG_TRAIN = "./data/images/train"
RUTA_IMG_TEST = "./data/images/test"
RUTA_LABELS_TRAIN = "./data/labels/train"
RUTA_LABELS_TEST = "./data/labels/test"
RUTA_MODELO = "./runs/pose/runs/cobb_v1-5/weights/best.pt"
RUTA_GT_TEST = "./data/Cobb_spinal-AI2024-test_gt.txt"

print("Rutas configuradas")
print(f"Modelo:{RUTA_MODELO}")

Python:  3.10.20  |  NumPy: 1.26.4  |  OpenCV: 4.9.0  |  PyTorch: 2.5.1+cu121
Rutas configuradas
Modelo:./runs/pose/runs/cobb_v1-5/weights/best.pt


---
## Dataset Spinal-AI2024

El dataset utilizado es **Spinal-AI2024** (disponible para descargar en https://github.com/Ernestchenchen/Spinal-AI2024), generado mediante el framework CurvNet mediante un pipeline iterativo de anotación supervisada por médicos.

### Características del dataset
- **20,000 imágenes** de rayos X en proyección anteroposterior (AP) de 512×512 píxeles
- **Partición:** 16,000 imágenes de entrenamiento (subsets 1–4) y 4,000 de prueba (subset 5)
- **Anotaciones:** coordenadas de las 4 esquinas de cada vértebra visible por imagen, en formato COCO JSON
- **Ground truth:** 3 ángulos de Cobb por imagen (PT, MT, TL), calculados con el algoritmo oficial del challenge AASCE-MICCAI 2019 a partir de los landmarks anotados (no medidos directamente por médicos sobre las radiografías).

### Limitaciones importantes del dataset

**Imágenes sintéticas**  
Las 20,000 imágenes fueron generadas por un modelo de inteligencia artificial a partir de un dataset clínico privado (Spinal2023). No son radiografías reales de pacientes sino imágenes artificialmente generadas que se ven como radiografías reales. Es por esto que se incluye una validación adicional con radiografías de entorno clínico real.

**Número inconsistente de vértebras por imagen**  
El dataset no tiene un número fijo de anotaciones por imagen. El análisis de las 16,000 imágenes de entrenamiento reveló la siguiente distribución:

| Vértebras anotadas | Imágenes | Porcentaje |
|---|---|---|
| < 15 | 39 | 0.2% |
| 15 | 611 | 3.8% |
| 16 | 3,043 | 19.0% |
| **17 (estándar)** | **6,736** | **42.1%** |
| 18 | 4,670 | 29.2% |
| 19 | 784 | 4.9% |
| > 19 | 20 | 0.1% |

Solo el **42.1%** de las imágenes tiene las 17 vértebras estándar (T1–L5). El 34.2% tiene más de 17 — lo que indica que en esas imágenes se anotaron vértebras cervicales (C6, C7) además de las torácicas y lumbares. El 23.7% tiene menos de 17, correspondiendo a columnas parcialmente visibles.

Esta inconsistencia se identificó durante el análisis del dataset y se corrigió en el cálculo del ángulo de Cobb usando un algoritmo de asignación de regiones anatómicas anclado desde la última vértebra (L5).

**Las anotaciones están en el campo `segmentation`, no en `keypoints`**  
A diferencia del formato COCO estándar para pose estimation, las coordenadas de las esquinas vertebrales en este dataset están almacenadas en el campo `segmentation` como polígonos de 4 puntos, no en el campo `keypoints`. Para resolver este problema se desarrolló un script de conversión que toma los datos en formato COCO y los convierte al formato requerido para utilizar YOLOv11-Pose.

### Estructura de las anotaciones

Cada anotación en el JSON tiene esta estructura:
```
  "image_id": 1,
  "bbox": [244, 35, 19, 14],
  "segmentation": [[246, 35, 262, 36, 245, 47, 262, 48]]
```

Donde `bbox` es `[x, y, ancho, alto]` en píxeles y `segmentation` contiene las 4 esquinas de la vértebra como `[x1, y1, x2, y2, x3, y3, x4, y4]`.

In [20]:
#Distribución completa de vértebras por imagen
from collections import Counter

RUTA_JSON = "./data/annotations/train.json"
with open(RUTA_JSON, "r") as f:
    datos = json.load(f)

anotaciones_por_imagen = {}
for anotacion in datos["annotations"]:
    img_id = anotacion["image_id"]
    if img_id not in anotaciones_por_imagen:
        anotaciones_por_imagen[img_id] = []
    anotaciones_por_imagen[img_id].append(anotacion)

#Cuenta cuántas vértebras tiene cada imagen del entrenamiento
todos_los_conteos = [len(anotaciones_por_imagen[img_id]) 
                     for img_id in anotaciones_por_imagen]
conteo_frecuencias = Counter(todos_los_conteos)

print("DISTRIBUCIÓN DE VÉRTEBRAS POR IMAGEN (16,000 imágenes)\n")
print(f"{'N° vértebras':>15} {'N° imágenes':>15} {'Porcentaje':>12}")
print("-" * 45)

for n_vert in sorted(conteo_frecuencias.keys()):
    n_imgs   = conteo_frecuencias[n_vert]
    porcentaje = n_imgs / len(todos_los_conteos) * 100
    barra    = "█" * int(porcentaje / 2)
    print(f"{n_vert:>15} {n_imgs:>15} {porcentaje:>11.1f}%  {barra}")

print("-" * 45)
print(f"{'TOTAL':>15} {len(todos_los_conteos):>15} {'100.0%':>12}")
print(f"\nEstadísticas:")
print(f"  Media:    {np.mean(todos_los_conteos):.2f} vértebras")
print(f"  Mediana:  {np.median(todos_los_conteos):.0f} vértebras")
print(f"  Moda:     {max(conteo_frecuencias, key=conteo_frecuencias.get)} vértebras")
print(f"  Mínimo:   {min(todos_los_conteos)}")
print(f"  Máximo:   {max(todos_los_conteos)}")

#Cuántas imágenes tienen exactamente 17 o más
print(f"\n  Con exactamente 17: {conteo_frecuencias.get(17, 0):,} "
      f"({conteo_frecuencias.get(17,0)/len(todos_los_conteos)*100:.1f}%)")
print(f"  Con más de 17:      "
      f"{sum(v for k,v in conteo_frecuencias.items() if k > 17):,} "
      f"({sum(v for k,v in conteo_frecuencias.items() if k > 17)/len(todos_los_conteos)*100:.1f}%)")
print(f"  Con menos de 17:    "
      f"{sum(v for k,v in conteo_frecuencias.items() if k < 17):,} "
      f"({sum(v for k,v in conteo_frecuencias.items() if k < 17)/len(todos_los_conteos)*100:.1f}%)")

DISTRIBUCIÓN DE VÉRTEBRAS POR IMAGEN (16,000 imágenes)

   N° vértebras     N° imágenes   Porcentaje
---------------------------------------------
              1               9         0.1%  
             11               4         0.0%  
             12               7         0.0%  
             13              19         0.1%  
             14              96         0.6%  
             15             611         3.8%  █
             16            3043        19.0%  █████████
             17            6736        42.1%  █████████████████████
             18            4670        29.2%  ██████████████
             19             784         4.9%  ██
             20              19         0.1%  
             21               1         0.0%  
---------------------------------------------
          TOTAL           15999       100.0%

Estadísticas:
  Media:    17.09 vértebras
  Mediana:  17 vértebras
  Moda:     17 vértebras
  Mínimo:   1
  Máximo:   21

  Con exactamente 17: 6,736 

---
## 3. Verificación visual de anotaciones del JSON original

Antes de hacer la conversión, verificamos visualmente las anotaciones directamente desde el archivo JSON original para confirmar que los puntos de segmentación corresponden correctamente a las esquinas de las vértebras. 

Se visualizan simultáneamente los puntos de segmentación y los bounding boxes de cada vértebra, usando un color distinto por vértebra para identificar visualmente la correspondencia entre puntos y estructura anatómica.

In [18]:
#VERIFICACIÓN VISUAL DE ANOTACIONES DEL JSON ORIGINAL
#Imagen de ejemplo
imagen_info = datos_train["images"][0]
img_id = imagen_info["id"]
file_name = imagen_info["file_name"]
w = imagen_info["width"]
h = imagen_info["height"]
anotaciones = anotaciones_por_imagen[img_id]
print(f"Imagen: {file_name} | "
      f"Dimensiones: {w}×{h}px | "
      f"Vértebras: {len(anotaciones)}")
img = cv2.imread(f"{RUTA_IMG_TRAIN}/{file_name}")
if len(img.shape) == 2:
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
img_kpts = img.copy()
img_boxes = img.copy()

def color_vertebra(i, n):
    hue = int(180 * i / n)
    color = cv2.cvtColor(
        np.uint8([[[hue, 255, 255]]]),
        cv2.COLOR_HSV2BGR
    )[0][0]
    return (int(color[0]), int(color[1]), int(color[2]))
n = len(anotaciones)

for i, anotacion in enumerate(anotaciones):
    color = color_vertebra(i, n)
    seg = anotacion["segmentation"][0]

#Dibuja los 4 keypoints
    for j in range(0, 8, 2):
        x = int(seg[j])
        y = int(seg[j+1])
        cv2.circle(img_kpts, (x, y), 3, color, -1)

#Centro de la vértebra para el número
    xs = [seg[j] for j in range(0, 8, 2)]
    ys = [seg[j] for j in range(1, 8, 2)]
    cx = int(np.mean(xs))
    cy = int(np.mean(ys))
    cv2.putText(img_kpts, str(i+1), (cx+8, cy),
                cv2.FONT_HERSHEY_SIMPLEX, 0.3, color, 1)

#Bounding box
    x, y, bw, bh = [int(v) for v in anotacion["bbox"]]
    cv2.rectangle(img_boxes, (x, y), (x+bw, y+bh), color, 1)
    cv2.putText(img_boxes, str(i+1), (x+bw+2, y+bh//2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.3, color, 1)

#Une las dos imágenes lado a lado
separador = np.ones((h, 10, 3), dtype=np.uint8) * 200
comparacion = np.hstack([img_kpts, separador, img_boxes])

cv2.putText(comparacion, "Segmentation points",
            (10, 20), cv2.FONT_HERSHEY_SIMPLEX,
            0.6, (255,255,255), 2)
cv2.putText(comparacion, "Bounding boxes",
            (w+20, 20), cv2.FONT_HERSHEY_SIMPLEX,
            0.6, (255,255,255), 2)

cv2.imwrite("./verificacion_json_original.png", comparacion)
print("Imagen guardada en ./verificacion_json_original.png")

Imagen: 000001.jpg | Dimensiones: 512×512px | Vértebras: 16
Imagen guardada en ./verificacion_json_original.png


---
## 4. Conversión de anotaciones COCO → YOLO-Pose

YOLOv11-Pose requiere las anotaciones en formato `.txt`, uno por imagen, con coordenadas normalizadas entre 0 y 1. El formato COCO original almacena las esquinas de las vértebras en el campo `segmentation` como coordenadas absolutas en píxeles, por lo que se desarrolló un script de conversión personalizado.

**Formato YOLO-Pose requerido**:
clase  cx  cy  bw  bh  x1  y1  v1  x2  y2  v2  x3  y3  v3  x4  y4  v4
Donde:
- `clase = 0` (vértebra, única clase del dataset)
- `cx, cy` = centro del bounding box normalizado ÷ dimensiones de imagen
- `bw, bh` = ancho y alto del bounding box normalizado
- `x1 y1 v1 ... x4 y4 v4` = coordenadas de las 4 esquinas normalizadas + visibilidad (`v=2`: visible)

In [14]:
def convertir_anotacion(bbox, segmentation, ancho_img, alto_img):
    #Bounding box
    cx = (bbox[0] + bbox[2] / 2) / ancho_img
    cy = (bbox[1] + bbox[3] / 2) / alto_img
    bw = bbox[2] / ancho_img
    bh = bbox[3] / alto_img
    #Keypoints
    puntos = segmentation[0] #Los datos se encuentran en una lista dentro de una lista
    x1, y1 = puntos[0] / ancho_img, puntos[1] / alto_img  # esquina superior izquierda
    x2, y2 = puntos[2] / ancho_img, puntos[3] / alto_img  # esquina superior derecha
    x3, y3 = puntos[4] / ancho_img, puntos[5] / alto_img  # esquina inferior izquierda
    x4, y4 = puntos[6] / ancho_img, puntos[7] / alto_img  # esquina inferior derecha
    v = 2
    #formato YOLO-Pose
    linea = (f"0 "
             f"{cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f} "
             f"{x1:.6f} {y1:.6f} {v} "
             f"{x2:.6f} {y2:.6f} {v} "
             f"{x3:.6f} {y3:.6f} {v} "
             f"{x4:.6f} {y4:.6f} {v}")
    
    return linea

#Prueba
anotacion_prueba = datos_train["annotations"][0]
imagen_prueba = datos_train["images"][0]

linea_prueba = convertir_anotacion(
    bbox = anotacion_prueba["bbox"],
    segmentation = anotacion_prueba["segmentation"],
    ancho_img = imagen_prueba["width"],
    alto_img = imagen_prueba["height"]
)

print("Resultado:")
print(linea_prueba)
print("Número de valores en la línea", len(linea_prueba.split()))
print("Valores esperados: 17")

Resultado:
0 0.495117 0.082031 0.037109 0.027344 0.480469 0.068359 2 0.511719 0.070312 2 0.478516 0.091797 2 0.511719 0.093750 2
Número de valores en la línea 17
Valores esperados: 17


In [17]:
#GENERACIÓN DE ARCHIVOS .TXT EN FORMATO YOLO-POSE

#Verifica que las carpetas de labels existen
os.makedirs(RUTA_LABELS_TRAIN, exist_ok=True)
os.makedirs(RUTA_LABELS_TEST,  exist_ok=True)
with open(RUTA_JSON_TEST, "r") as f:
    datos_test = json.load(f)

print(f"JSON test cargado: {len(datos_test['images']):,} imágenes")

def generar_labels(datos_json, ruta_labels, nombre_split):
    #Construye el índice de búsqueda
    anotaciones_idx = {}
    for anotacion in datos_json["annotations"]:
        img_id = anotacion["image_id"]
        if img_id not in anotaciones_idx:
            anotaciones_idx[img_id] = []
        anotaciones_idx[img_id].append(anotacion)
    archivos_creados = 0
    sin_anotaciones = 0

    for imagen in datos_json["images"]:
        img_id = imagen["id"]
        file_name = imagen["file_name"]
        ancho_img = imagen["width"]
        alto_img  = imagen["height"]

        #Nombre del .txt = mismo nombre que la imagen
        nombre_txt = file_name.replace(".jpg", ".txt") \
                               .replace(".png", ".txt")
        ruta_txt = os.path.join(ruta_labels, nombre_txt)

        anotaciones = anotaciones_idx.get(img_id, [])
        
        #Imagen sin anotaciones crea un archivo vacío
        if len(anotaciones) == 0:
            open(ruta_txt, "w").close()
            sin_anotaciones += 1
            continue

        #Escribe una línea por vértebra
        with open(ruta_txt, "w") as f:
            for anotacion in anotaciones:
                linea = convertir_anotacion(
                    bbox = anotacion["bbox"],
                    segmentation = anotacion["segmentation"],
                    ancho_img = ancho_img,
                    alto_img = alto_img
                )
                f.write(linea + "\n")
        archivos_creados += 1

    print(f"Split {nombre_split}:")
    print(f"Archivos .txt creados: {archivos_creados:,}")
    print(f"Imágenes sin anotaciones: {sin_anotaciones}")

#Verifica si los labels ya existen para no regenerarlos
n_train = len(os.listdir(RUTA_LABELS_TRAIN))
n_test  = len(os.listdir(RUTA_LABELS_TEST))

if n_train >= 16000 and n_test >= 4000:
    print(f"Labels ya existen:")
    print(f"Train: {n_train:,} archivos")
    print(f"Test: {n_test:,} archivos")
    print(f"No es necesario regenerarlos.")
else:
    print("Generando archivos .txt...")
    #Carga JSON de test si no está en memoria
    if "datos_test" not in dir():
        with open(RUTA_JSON_TEST, "r") as f:
            datos_test = json.load(f)

generar_labels(datos_train, RUTA_LABELS_TRAIN, "train")
generar_labels(datos_test,  RUTA_LABELS_TEST,  "test")
print("\n Conversión completada")

JSON test cargado: 4,000 imágenes
Labels ya existen:
Train: 16,000 archivos
Test: 4,000 archivos
No es necesario regenerarlos.
Split train:
Archivos .txt creados: 15,999
Imágenes sin anotaciones: 1
Split test:
Archivos .txt creados: 4,000
Imágenes sin anotaciones: 0

 Conversión completada


---
## 5. Verificación visual de anotaciones en formato YOLO

Se verifica que la conversión al formato YOLO-Pose fue correcta dibujando los keypoints de los archivos `.txt` generados sobre las imágenes originales. Si la conversión es correcta, los puntos deben caer exactamente sobre las esquinas de las vértebras.

**Código de colores:**
- Azul: esquina superior izquierda (sup_izq)
- Rojo: esquina superior derecha (sup_der)
- Amarillo: esquina inferior izquierda (inf_izq)
- Verde: esquina inferior derecha (inf_der)

In [21]:
# VERIFICACIÓN VISUAL DE LAS ANOTACIONES YOLO-POSE

RUTA_IMG_VERIFICAR = f"{RUTA_IMG_TRAIN}/000001.jpg"
RUTA_TXT_VERIFICAR = f"{RUTA_LABELS_TRAIN}/000001.txt"

#Carga la imagen
img = cv2.imread(RUTA_IMG_VERIFICAR)
if len(img.shape) == 2:
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
h, w = img.shape[:2]

#Lee el archivo .txt generado
with open(RUTA_TXT_VERIFICAR, "r") as f:
    lineas = f.readlines()
print(f"Imagen: 000001.jpg  |  Vértebras en .txt: {len(lineas)}")

#Colores por esquina
colores_bgr = [
    (255, 0,   0  ), #azul
    (0,   0,   255), #rojo
    (0,   255, 255), #amarillo
    (0,   255, 0  ), #verde
]
nombres_esquinas = ["sup_izq", "sup_der", "inf_izq", "inf_der"]

for linea in lineas:
    valores = linea.strip().split()
    #Keypoints empiezan en índice 5 con 3 valores: x, y, visibilidad
    for i in range(4):
        idx = 5 + i * 3
        x = int(float(valores[idx])     * w)
        y = int(float(valores[idx + 1]) * h)
        cv2.circle(img, (x, y), 3, colores_bgr[i], -1)

#Leyenda de colores en la imagen
for i, (nombre, color) in enumerate(
        zip(nombres_esquinas, colores_bgr)):
    cv2.rectangle(img, (5, 8 + i*18),
                  (18, 20 + i*18), color, -1)
    cv2.putText(img, nombre, (22, 18 + i*18),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4, (255,255,255), 1)

cv2.imwrite("./verificacion_labels_yolo.png", img)
print("Imagen guardada en ./verificacion_labels_yolo.png")

Imagen: 000001.jpg  |  Vértebras en .txt: 16
Imagen guardada en ./verificacion_labels_yolo.png


---
## 6. Configuración del entrenamiento

### 6.1 Archivo de configuración YAML

YOLOv11-Pose necesita un archivo YAML que especifica las rutas del dataset, el número de clases y la estructura de los keypoints. Este archivo es obligatorio ya que sin él el entrenador no sabe dónde buscar las imágenes ni cuántos keypoints predecir por objeto [1].
El parámetro `kpt_shape: [4, 3]` indica que cada vértebra tiene 4 keypoints y cada keypoint tiene 3 valores (x, y, visibilidad).

In [22]:
#VERIFICACIÓN DEL ARCHIVO DE CONFIGURACIÓN YAML
import yaml
RUTA_YAML = "./spine_pose.yaml"
#Verifica que el archivo existe
if not os.path.exists(RUTA_YAML):
    print("El archivo spine_pose.yaml no existe.")
else:
    with open(RUTA_YAML, "r") as f:
        config = yaml.safe_load(f)

    print("Contenido de spine_pose.yaml:")
    print("-" * 35)
    for llave, valor in config.items():
        print(f"  {llave}: {valor}")
    print("-" * 35)
    print("Archivo YAML verificado correctamente")

Contenido de spine_pose.yaml:
-----------------------------------
  path: .
  train: data/images/train
  val: data/images/test
  kpt_shape: [4, 3]
  names: {0: 'vertebra'}
-----------------------------------
Archivo YAML verificado correctamente


---
## 7. Entrenamiento del modelo

### 7.1 Arquitectura YOLOv11n-Pose

Se utiliza la variante nano de YOLOv11-Pose, seleccionada con base en el estudio comparativo de Rios et al. (2025), que evaluó directamente YOLOv8-Pose, YOLOv11-Pose y Detectron2 para detección de vértebras con keypoints, encontrando que YOLOv11n-Pose provee el mejor balance entre precisión de keypoints y eficiencia de inferencia para esta tarea específica.

La arquitectura consta de tres componentes:
- **Backbone CSPDarknet** extrae características a múltiples escalas
- **Neck FPN+PAN** combina información semántica y de localización
- **Head Pose** predice bounding boxes y 4 keypoints por vértebra simultáneamente
- 
**Parámetros:** 2.6M  |  **VRAM requerida:** ~2GB
  
*Recomendación*: El entrenamiento puede tardar mucho dependiendo de las características del hardware utilizado. Puede resultar conveniente entrenar al modelo en Google Colab para agilizar el proceso y evitar problemas de sobrecalentamiento. 

### 7.2 Transfer Learning
Se parte de pesos pre-entrenados en COCO (`yolo11n-pose.pt`).Las capas iniciales ya saben detectar bordes y formas generales solo las capas finales se re-entrenan para detectar vértebras con 4 keypoints.

### 7.3 Hiperparámetros del primer entrenamiento (versión 0.1e50)

| Parámetro | Valor | Justificación |
|---|---|---|
| `epochs` | 50 | Convergencia máxima que se puede lograr con la GPU local |
| `imgsz` | 512 | Mismo tamaño que las imágenes del dataset |
| `batch` | 8 | Máximo para GPU |
| `patience` | 15 | Early stopping [6] |
| `flipud` | 0.0 | Sin volteo vertical: anatómicamente inválido |
| `fliplr` | 0.5 | Volteo horizontal: columna en espejo válida |
| `degrees` | 5.0° | Rotación: simula radiografías inclinadas |
| `scale` | 0.2 | Zoom: simula distancias de adquisición |
| `hsv_v` | 0.5 | Variación de brillo: distintos equipos |

El primer entrenamiento se realizó en un GPU local
(NVIDIA GeForce GTX 1650 Ti, 4GB VRAM).

In [24]:
#Primer entrenamiento en GPU local
# El modelo entrenado está disponible en: runs/pose/runs/cobb_v1/weights/best.pt junto con los resultados del entrenamiento.
from ultralytics import YOLO

model = YOLO("yolo11n-pose.pt")

#results = model.train(
#    data     = "./spine_pose.yaml",
#    epochs   = 50,
#    imgsz    = 512,
#    batch    = 8,
#    patience = 15,
#    device   = 0,
#    project  = "./runs",
#    name     = "cobb_v1",
#    flipud   = 0.0,
#    fliplr   = 0.5,
#    degrees  = 5.0,
#    scale    = 0.2,
#    workers  = 0,
#)

#print("Entrenamiento terminado")
#print("Mejor modelo en: ./runs/cobb_v1/weights/best.pt")
#El entrenamiento ya fue ejecutado 
RUTA_MODELO_e50 = "./runs/pose/runs/cobb_v1/weights/best.pt"

if os.path.exists(RUTA_MODELO_e50):
    print(f" Modelo entrenado encontrado en:")
    print(f"{RUTA_MODELO_e50}")
    
    #Muestra información del modelo
    modelo_info = YOLO(RUTA_MODELO_e50)
    print(f"\nInformación del modelo versión 0.1e50:")
    print(f"Tipo: YOLOv11n-Pose")
    print(f"Parámetros: 2,662,263")
    print(f"Tarea: pose estimation")
else:
    print(f" Modelo no encontrado en {RUTA_MODELO_e50}")
    print(f"Ejecuta el entrenamiento primero.")

 Modelo entrenado encontrado en:
./runs/pose/runs/cobb_v1/weights/best.pt

Información del modelo versión 0.1e50:
Tipo: YOLOv11n-Pose
Parámetros: 2,662,263
Tarea: pose estimation


### 7.4 Hiperparámetros del segundo entrenamiento (versión 0.1e100)

| Parámetro | Valor | Justificación |
|---|---|---|
| `epochs` | 100 | Convergencia completa |
| `imgsz` | 512 | Mismo tamaño que las imágenes del dataset |
| `batch` | 16 | Máximo para GPU T4 (Colab) |
| `patience` | 20 | Early stopping [6] |
| `flipud` | 0.0 | Sin volteo vertical: anatómicamente inválido |
| `fliplr` | 0.5 | Volteo horizontal: columna en espejo válida |
| `degrees` | 5.0° | Rotación: simula radiografías inclinadas |
| `scale` | 0.2 | Zoom: simula distancias de adquisición |
| `hsv_v` | 0.5 | Variación de brillo: distintos equipos |
| `save_period` | 10 | Checkpoint cada 10 épocas |

El entrenamiento se realizó en Google Colab con GPU Tesla T4

In [ ]:
#ENTRENAMIENTO DEL MODELO
#NOTA: Este código se ejecutó en Google Colab con GPU Tesla T4.
#El modelo entrenado está disponible en:
#runs/pose/runs/cobb_v1-2/weights/best.pt

from ultralytics import YOLO

model = YOLO("yolo11n-pose.pt")

# Entrenamiento
results = model.train(
    data        = "./spine_pose.yaml",
    epochs      = 100,
    imgsz       = 512,
    batch       = 16,
    patience    = 20,
    device      = 0,
    project     = "./runs",
    name        = "cobb_v1",
    flipud      = 0.0,
    fliplr      = 0.5,
    degrees     = 5.0,
    scale       = 0.2,
    workers     = 2,
    save_period = 10,
    hsv_v       = 0.5,
)

# El entrenamiento ya fue ejecutado.


RUTA_MODELO_e100 = "./runs/pose/runs/cobb_v1-5/weights/best.pt"

if os.path.exists(RUTA_MODELO_e100):
    print(f"Modelo entrenado encontrado en:")
    print(f"{RUTA_MODELO_e100}")
    
    # Muestra información del modelo
    modelo_info = YOLO(RUTA_MODELO_e100)
    print(f"\nInformación del modelo:")
    print(f"Tipo:YOLOv11n-Pose")
    print(f"Parámetros: 2,662,263")
    print(f"Tarea: pose estimation")
else:
    print(f"Modelo no encontrado en {RUTA_MODELO_e100}")
    print(f"Ejecuta el entrenamiento primero.")

---
## 8. Resultados del entrenamiento

### 8.1 Resultados del primer entrenamiento
Las curvas de pérdida y métricas se guardaron automáticamente en `runs/pose/runs/cobb_v1/results.png` durante el entrenamiento hecho desde la laptop.

### Interpretación de las curvas

**Curvas de pérdida (loss):** Todas las pérdidas de entrenamiento y validación descienden consistentemente sin divergencia entre ambas, esto indica que el modelo aprendió correctamente sin sobreajuste (*overfitting*).

**Métricas de detección de bounding boxes (B):**
- `precision(B)` ≈ 0.993, de cada 100 vértebras detectadas, 99.3 son correctas
- `recall(B)` ≈ 0.992, de cada 100 vértebras reales, el modelo encontró 99.2

**Métricas de pose/keypoints (P):**
- `mAP50-95(P)` ≈ 0.990, métrica principal de keypoints, evaluada a múltiples umbrales de distancia. Un valor de 0.990 indica que el 99% de los keypoints predichos están dentro de la distancia aceptable respecto al ground truth.

In [30]:
#RESULTADOS DEL ENTRENAMIENTO v0.1e50
RUTA_RESULTS = "./runs/pose/runs/cobb_v1/results.png"
RUTA_VAL_PRED = "./runs/pose/runs/cobb_v1/val_batch0_pred.jpg"

if os.path.exists(RUTA_RESULTS):
    # Carga y guarda la imagen de resultados
    img_results = cv2.imread(RUTA_RESULTS)
    h, w = img_results.shape[:2]
    print(f"Curvas de entrenamiento cargadas")
    print(f"Dimensiones: {w}×{h}px")
    cv2.imwrite("./resultados_entrenamiento.png", img_results)
    print("Guardada en ./resultados_entrenamiento.png")
else:
    print(f"No se encontró {RUTA_RESULTS}")

#Muestra una imagen de validación con predicciones
if os.path.exists(RUTA_VAL_PRED):
    img_val = cv2.imread(RUTA_VAL_PRED)
    cv2.imwrite("./validacion_predicciones.png", img_val)
    print("Imagen de validación guardada en " "./validacion_predicciones.png")

#Carga y muestra las métricas numéricas del CSV
import csv
RUTA_CSV = "./runs/pose/runs/cobb_v1/results.csv"

if os.path.exists(RUTA_CSV):
    with open(RUTA_CSV, "r") as f:
        reader = csv.DictReader(f)
        filas  = list(reader)

    #Última época con mejores métricas
    ultima = filas[-1]
    print(f"\nMÉTRICAS FINALES v0.1e50 (última época)")
    for key, val in ultima.items():
        key_limpio = key.strip()
        val_limpio = val.strip()
        if val_limpio:
            try:
                print(f"  {key_limpio:<35} {float(val_limpio):.4f}")
            except ValueError:
                pass

    #Mejor época según mAP50-95 de pose
    key_map = "metrics/mAP50-95(P)"
    mejor_epoca = max(filas,
        key=lambda r: float(r[key_map].strip())
                      if r[key_map].strip() else 0)
    print(f"\n  Mejor época: "
          f"{mejor_epoca['epoch'].strip()}/50")
    print(f"  mAP50-95(P): "
          f"{float(mejor_epoca[key_map].strip()):.4f}")
else:
    print(f"No se encontró {RUTA_CSV}")

Curvas de entrenamiento cargadas
Dimensiones: 4000×1200px
Guardada en ./resultados_entrenamiento.png
Imagen de validación guardada en ./validacion_predicciones.png

MÉTRICAS FINALES v0.1e50 (última época)
  epoch                               50.0000
  time                                35337.8000
  train/box_loss                      0.5812
  train/pose_loss                     0.0491
  train/kobj_loss                     0.0020
  train/cls_loss                      0.2529
  train/dfl_loss                      0.7998
  metrics/precision(B)                0.9878
  metrics/recall(B)                   0.9920
  metrics/mAP50(B)                    0.9930
  metrics/mAP50-95(B)                 0.8577
  metrics/precision(P)                0.9886
  metrics/recall(P)                   0.9928
  metrics/mAP50(P)                    0.9931
  metrics/mAP50-95(P)                 0.9914
  val/box_loss                        0.5800
  val/pose_loss                       0.0490
  val/kobj_loss          

---
### 8.2 Resultados del segundo entrenamiento v1e100 (Colab, 58 épocas)

Las curvas de pérdida y métricas del segundo entrenamiento se guardaron en `runs/pose/runs/cobb_v1e100/results.png`.

In [64]:
# =============================================================
# RESULTADOS DEL ENTRENAMIENTO v1e100 (Colab)
# =============================================================

import cv2
import csv

RUTA_RESULTS_V2 = "./runs/pose/runs/cobb_v1e100/results.csv"
RUTA_CSV_V2     = "./runs/pose/runs/cobb_v1e100/results.csv"


# Métricas numéricas
if os.path.exists(RUTA_CSV_V2):
    with open(RUTA_CSV_V2, "r") as f:
        filas = list(csv.DictReader(f))

    ultima = filas[-1]
    print(f"\nMétricas finales v1e100 (época {len(filas)}):")

    metricas_mostrar = [
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)",
        "metrics/precision(P)",
        "metrics/recall(P)",
        "metrics/mAP50(P)",
        "metrics/mAP50-95(P)",
    ]
    for key in ultima:
        key_limpio = key.strip()
        if key_limpio in metricas_mostrar:
            try:
                val = float(ultima[key].strip())
                print(f"  {key_limpio:<30} {val:.4f}")
            except ValueError:
                pass
else:
    print(f"\n⚠️ No se encontró {RUTA_CSV_V2}")

# Comparación de métricas finales v1 vs v1e100
print(f"\n=== COMPARACIÓN DE MÉTRICAS DE ENTRENAMIENTO ===")
print(f"{'Métrica':<30} {'v1 (50 ep)':>12} {'v1e100 (100 ep)':>15}")
print(f"{'-'*58}")

RUTA_CSV_V1 = "./runs/pose/runs/cobb_v1/results.csv"

if os.path.exists(RUTA_CSV_V1) and os.path.exists(RUTA_CSV_V2):
    with open(RUTA_CSV_V1) as f:
        ultima_v1 = list(csv.DictReader(f))[-1]
    with open(RUTA_CSV_V2) as f:
        ultima_v2 = list(csv.DictReader(f))[-1]

    for key in ultima_v1:
        key_limpio = key.strip()
        if key_limpio in metricas_mostrar:
            try:
                val_v1 = float(ultima_v1[key].strip())
                val_v2 = float(ultima_v2[key].strip())
                mejor  = "✅" if val_v1 >= val_v2 else "  "
                print(f"  {key_limpio:<28} "
                      f"{val_v1:>11.4f}  "
                      f"{val_v2:>14.4f} {mejor}")
            except (ValueError, KeyError):
                pass
else:
    print("  Descarga los archivos results.csv de ambos "
          "experimentos para ver la comparación")


Métricas finales v1e100 (época 58):
  metrics/precision(B)           0.9865
  metrics/recall(B)              0.9920
  metrics/mAP50(B)               0.9929
  metrics/mAP50-95(B)            0.8537
  metrics/precision(P)           0.9874
  metrics/recall(P)              0.9930
  metrics/mAP50(P)               0.9929
  metrics/mAP50-95(P)            0.9911

=== COMPARACIÓN DE MÉTRICAS DE ENTRENAMIENTO ===
Métrica                          v1 (50 ep) v1e100 (100 ep)
----------------------------------------------------------
  metrics/precision(B)              0.9878          0.9865 ✅
  metrics/recall(B)                 0.9920          0.9920 ✅
  metrics/mAP50(B)                  0.9930          0.9929 ✅
  metrics/mAP50-95(B)               0.8577          0.8537 ✅
  metrics/precision(P)              0.9886          0.9874 ✅
  metrics/recall(P)                 0.9928          0.9930   
  metrics/mAP50(P)                  0.9931          0.9929 ✅
  metrics/mAP50-95(P)               0.9914    

---
## 10. Cálculo del ángulo de Cobb

Una vez obtenidos los keypoints predichos por el modelo se aplica un pipeline geométrico de cinco pasos para calcular los tres ángulos de Cobb (PT, MT y TL).

### Pipeline geométrico

**Paso 1: Ordenamiento craneocaudal**
Las vértebras se ordenan de T1 a L5 por su coordenada `Y` promedio. En imágenes AP, `Y` menor = más arriba = más cerca de T1.

**Paso 2: Ángulo de inclinación de cada vértebra**
Para cada vértebra se calcula el ángulo de su plataforma superior (línea entre sup_izq y sup_der) respecto a la horizontal mediante `arctan2(dy, dx)`.

**Paso 3: Asignación de regiones anatómicas**
Las regiones se asignan anclando desde L5 que es la última vértebra siempre, para manejar correctamente imágenes con vértebras cervicales extra las cuales son el 34.2% del dataset:

| Región | Vértebras | Asignación |
|---|---|---|
| PT | T1–T4 | últimas 17 a últimas 14 |
| MT | T5–T12 | últimas 13 a últimas 6 |
| TL | L1–L5 | últimas 5 a última |

**Paso 4: Detección de *end vertebrae* por derivada discreta**
Se calcula Δθᵢ = θᵢ₊₁ - θᵢ dentro de cada región. Un cambio de signo en Δθᵢ · Δθᵢ₊₁ < 0 identifica la vértebra extrema [3].

**Paso 5 — Ángulo de Cobb**
Cobb_región = |θ_max - θ_min| dentro de cada región.

In [37]:
#CARGA DEL MODELO Y EXTRACCIÓN DE KEYPOINTS (imagen de prueba)
from ultralytics import YOLO

#Carga el modelo entrenado
modelo    = YOLO(RUTA_MODELO)

#Corre inferencia sobre una imagen de prueba
RUTA_IMG_PRUEBA = f"{RUTA_IMG_TEST}/016003.jpg"
resultados      = modelo(RUTA_IMG_PRUEBA, conf=0.3, verbose=False)
r               = resultados[0]
keypoints       = r.keypoints.xy.cpu().numpy()

print(f"✅ Modelo cargado y keypoints extraídos")
print(f"   Vértebras detectadas: {len(keypoints)}")

✅ Modelo cargado y keypoints extraídos
   Vértebras detectadas: 19


In [44]:
#CÁLCULO DEL ÁNGULO DE COBB
def asignar_regiones(n_vertebras):
    #Las últimas 5 siempre son L1-L5
    idx_tl_ini = max(0, n_vertebras - 5)
    #Las 8 anteriores son T5-T12
    idx_mt_ini = max(0, idx_tl_ini - 8)
    #Las 4 anteriores son T1-T4
    idx_pt_ini = max(0, idx_mt_ini - 4)

    return {
        "cervicales": list(range(0, idx_pt_ini)),
        "PT": list(range(idx_pt_ini, idx_mt_ini)),
        "MT": list(range(idx_mt_ini, idx_tl_ini)),
        "TL": list(range(idx_tl_ini, n_vertebras))
    }

def calcular_angulo_cobb_v2(keypoints_vertebras):
    n = len(keypoints_vertebras)

    #Ordena de arriba hacia abajo
    centros_y = [np.mean(kpts[:, 1])
                 for kpts in keypoints_vertebras]
    orden = np.argsort(centros_y)
    vertebras = keypoints_vertebras[orden]

    #Ángulo de inclinación de cada vértebra
    #Plataforma superior = línea entre kpts[0] y kpts[1]
    angulos = []
    for kpts in vertebras:
        sup_izq = kpts[0]
        sup_der = kpts[1]
        dx = sup_der[0] - sup_izq[0]
        dy = sup_der[1] - sup_izq[1]
        angulo = np.degrees(np.arctan2(dy, dx))
        angulos.append(angulo)
    angulos = np.array(angulos)

    #Asigna regiones anclando desde L5
    regiones = asignar_regiones(n)

    #Derivada discreta y ángulo Cobb
    def cobb_region(indices, nombre):
        if len(indices) < 2:
            return 0.0, {
                "region":     nombre,
                "n_vert":     len(indices),
                "confiable":  False,
                "razon":      "Menos de 2 vértebras"
            }

        ang = angulos[indices]
        derivada = np.diff(ang)

        #Detecta cambio de signo e identifica la end vertebra
        hay_inflexion = any(
            derivada[i] * derivada[i+1] < 0
            for i in range(len(derivada)-1)
        )

        #Ángulo de Cobb = diferencia entre max y min
        angulo_cobb = float(abs(np.max(ang) - np.min(ang)))

        return angulo_cobb, {
            "region":        nombre,
            "n_vert":        len(indices),
            "hay_inflexion": hay_inflexion,
            "confiable":     len(indices) >= 3,
            "razon":         "OK" if len(indices) >= 3
                             else "Solo 2 vértebras"
        }

    pt, info_pt = cobb_region(regiones["PT"], "PT")
    mt, info_mt = cobb_region(regiones["MT"], "MT")
    tl, info_tl = cobb_region(regiones["TL"], "TL")

    info = {
        "n_detectadas": n,
        "n_cervicales": len(regiones["cervicales"]),
        "PT": info_pt,
        "MT": info_mt,
        "TL": info_tl,
        "advertencias": []
    }

    if len(regiones["cervicales"]) > 0:
        info["advertencias"].append(
            f"{len(regiones['cervicales'])} vértebra(s) "
            f"cervical(es) ignoradas"
        )
    if n < 15:
        info["advertencias"].append(
            f"Solo {n} vértebras — resultado puede "
            f"ser impreciso"
        )

    return abs(pt), abs(mt), abs(tl), info


def clasificar_escoliosis(angulo_mt):
    if angulo_mt < 10:
        return "Normal (<10°)"
    elif angulo_mt < 25:
        return "Leve (10°–25°) — seguimiento periódico"
    elif angulo_mt < 40:
        return "Moderada (25°–40°) — considerar ortesis"
    else:
        return "Severa (>40°) — evaluación quirúrgica"


#Prueba con la imagen 016001.jpg
pt, mt, tl, info = calcular_angulo_cobb_v2(keypoints)

print("PRUEBA: imagen 016003.jpg con modelo versión 0.1e50 \n")
print(f"Vértebras detectadas: {info['n_detectadas']}")
print(f"Cervicales ignoradas: {info['n_cervicales']}")
print(f"\nÁngulos predichos:")
print(f"  PT: {pt:.2f}°  "
      f"({'✅' if info['PT']['confiable'] else '⚠️'}  "
      f"{info['PT']['n_vert']} vértebras)")
print(f"  MT: {mt:.2f}°  "
      f"({'✅' if info['MT']['confiable'] else '⚠️'}  "
      f"{info['MT']['n_vert']} vértebras)")
print(f"  TL: {tl:.2f}°  "
      f"({'✅' if info['TL']['confiable'] else '⚠️'}  "
      f"{info['TL']['n_vert']} vértebras)")
print(f"\nClasificación: {clasificar_escoliosis(mt)}")

if info["advertencias"]:
    print(f"\nAdvertencias:")
    for adv in info["advertencias"]:
        print(f"  ⚠️ {adv}")

PRUEBA: imagen 016003.jpg con modelo versión 0.1e50 

Vértebras detectadas: 19
Cervicales ignoradas: 2

Ángulos predichos:
  PT: 3.96°  (✅  4 vértebras)
  MT: 10.00°  (✅  8 vértebras)
  TL: 14.27°  (✅  5 vértebras)

Clasificación: Normal (<10°)

Advertencias:
  ⚠️ 2 vértebra(s) cervical(es) ignoradas


---
## 11. Evaluación cuantitativa

Se evalúa el modelo sobre las 4,000 imágenes del set de prueba del dataset Spinal-AI2024, comparando los ángulos predichos contra el ground truth calculado con el algoritmo oficial MATLAB del challenge AASCE-MICCAI 2019.

### Métricas utilizadas

**MAE (Mean Absolute Error)**:  métrica principal en grados:
$$MAE = \frac{1}{N} \sum_{i=1}^{N} |\hat{y}_i - y_i|$$

Un MAE < 5° se considera clínicamente aceptable, equivalente a la variabilidad interobservador humana de 3–10°.

**SMAPE (Symmetric Mean Absolute Percentage Error)**: métrica oficial del challenge AASCE-MICCAI 2019, permite
comparación directa con la literatura:

$$SMAPE = \frac{1}{N} \sum_{i=1}^{N}
\frac{|\hat{y}_i - y_i|}{(|\hat{y}_i| + |y_i|)/2} \times 100\%$$

Se prefiere sobre el MAPE porque es simétrica y está acotada entre 0% y 200%, evitando valores infinitos cuando el ángulo real es cercano a cero.

**Nota sobre PT = 0.0°:** 394 imágenes (9.9%) tienen PT = 0.0° en el ground truth, indicando ausencia de curva torácica proximal. El MAE y SMAPE de PT se reportan incluyendo y excluyendo estos casos para una evaluación más justa del modelo.

In [42]:
#CARGA DEL GROUND TRUTH
ground_truth = {}

with open(RUTA_GT_TEST, "r") as f:
    for linea in f.readlines():
        partes = linea.strip().split(",")
        if len(partes) == 4:
            ground_truth[partes[0]] = [
                float(partes[1]),  # PT
                float(partes[2]),  # MT
                float(partes[3])   # TL
            ]

print(f"Ground truth cargado: {len(ground_truth):,} imágenes")
print(f"\nEjemplo — 016001.jpg:")
gt = ground_truth["016001.jpg"]
print(f"  PT: {gt[0]}°  MT: {gt[1]}°  TL: {gt[2]}°")

Ground truth cargado: 4,000 imágenes

Ejemplo — 016001.jpg:
  PT: 0.0°  MT: 4.09°  TL: 12.45°


In [43]:
#EVALUACIÓN SOBRE 4,000 IMÁGENES DE PRUEBA
#NOTA: Esta celda tarda varios minutos en ejecutarse.

import time

imagenes = sorted([
    f for f in os.listdir(RUTA_IMG_TEST)
    if f.endswith(".jpg") and f in ground_truth
])

#Listas para acumular errores
errores_pt = []
errores_mt = []
errores_tl = []
errores_pt_validos = []  # excluye PT=0.0° en ground truth

#Para análisis por severidad
resultados_detalle = []
con_cervicales = 0
sin_deteccion = 0

print(f"Evaluando {len(imagenes):,} imágenes con modelo versión 0.1e50...")
inicio = time.time()

for i, nombre_img in enumerate(imagenes):
    if i % 500 == 0:
        print(f"  {i}/{len(imagenes)}...")

    ruta_img = os.path.join(RUTA_IMG_TEST, nombre_img)
    resultados = modelo(ruta_img, conf=0.3, verbose=False)
    r = resultados[0]

    if r.keypoints is None or len(r.keypoints.xy) < 2:
        sin_deteccion += 1
        continue

    kpts = r.keypoints.xy.cpu().numpy()
    pt, mt, tl, info = calcular_angulo_cobb_v2(kpts)
    gt = ground_truth[nombre_img]
    pt_gt, mt_gt, tl_gt = gt[0], gt[1], gt[2]

    #Errores absolutos
    e_pt = abs(pt - pt_gt)
    e_mt = abs(mt - mt_gt)
    e_tl = abs(tl - tl_gt)

    errores_pt.append(e_pt)
    errores_mt.append(e_mt)
    errores_tl.append(e_tl)

    if pt_gt > 0.0:
        errores_pt_validos.append(e_pt)

    if info["n_cervicales"] > 0:
        con_cervicales += 1

    #Guarda detalle para análisis por severidad y SMAPE
    resultados_detalle.append({
        "pt_pred": pt,  "mt_pred": mt,  "tl_pred": tl,
        "pt_gt":   pt_gt, "mt_gt": mt_gt, "tl_gt":  tl_gt,
        "e_pt":    e_pt,  "e_mt":  e_mt,  "e_tl":   e_tl,
    })

tiempo = time.time() - inicio

#Función SMAPE
def smape(predichos, reales):
    valores = []
    for pred, real in zip(predichos, reales):
        denominador = (abs(pred) + abs(real)) / 2
        if denominador > 0:
            valores.append(abs(pred - real) / denominador * 100)
    return np.mean(valores) if valores else 0.0

#Resultados
pt_preds = [r["pt_pred"] for r in resultados_detalle]
mt_preds = [r["mt_pred"] for r in resultados_detalle]
tl_preds = [r["tl_pred"] for r in resultados_detalle]
pt_gts = [r["pt_gt"]   for r in resultados_detalle]
mt_gts = [r["mt_gt"]   for r in resultados_detalle]
tl_gts = [r["tl_gt"]   for r in resultados_detalle]

smape_pt = smape(pt_preds, pt_gts)
smape_mt = smape(mt_preds, mt_gts)
smape_tl = smape(tl_preds, tl_gts)
smape_total = smape(
    pt_preds + mt_preds + tl_preds,
    pt_gts   + mt_gts   + tl_gts
)

print(f"RESULTADOS FINALES DE EVALUACIÓN DEL MODELO VERSIÓN 0.1e50")
print(f"Imágenes evaluadas: {len(errores_pt):,}")
print(f"Sin detección: {sin_deteccion}")
print(f"Con cervicales ignoradas: {con_cervicales:,} "
      f"({con_cervicales/len(errores_pt)*100:.1f}%)")
print(f"Tiempo total: {tiempo/60:.1f} min")

print(f"\n{'Ángulo':<8} {'MAE (°)':>10} {'SMAPE (%)':>12}")
print(f"{'-'*32}")
print(f"{'PT':<8} {np.mean(errores_pt):>10.2f} "
      f"{smape_pt:>11.2f}%")
print(f"{'MT':<8} {np.mean(errores_mt):>10.2f} "
      f"{smape_mt:>11.2f}%")
print(f"{'TL':<8} {np.mean(errores_tl):>10.2f} "
      f"{smape_tl:>11.2f}%")
print(f"{'-'*32}")
print(f"{'Total':<8} "
      f"{np.mean(errores_pt+errores_mt+errores_tl):>10.2f} "
      f"{smape_total:>11.2f}%")

print(f"\nMAE PT excluyendo ground truth = 0.0°:")
print(f"  {np.mean(errores_pt_validos):.2f}°  "
      f"({len(errores_pt_validos):,} imágenes)")

print(f"\nReferencia clínica: MAE < 5° = clínicamente aceptable")
print(f"Referencia literatura: SMAPE ~21.71% (mejor método "
      f"AASCE2019, Wang et al. 2021)")

#Análisis por severidad
rangos = [
    ("Normal (<10°)",      0,  10),
    ("Leve (10–25°)",     10,  25),
    ("Moderada (25–40°)", 25,  40),
    ("Severa (>40°)",     40, 999),
]

print(f"\nMAE MT por severidad de la escoliosis:")
print(f"  {'Rango':<22} {'MAE':>8} {'SMAPE':>10} {'N':>6}")
print(f"  {'-'*50}")
for nombre_rango, min_a, max_a in rangos:
    subset = [r for r in resultados_detalle
              if min_a <= r["mt_gt"] < max_a]
    if subset:
        mae_s   = np.mean([r["e_mt"] for r in subset])
        smape_s = smape(
            [r["mt_pred"] for r in subset],
            [r["mt_gt"]   for r in subset]
        )
        print(f"  {nombre_rango:<22} "
              f"{mae_s:>7.2f}° "
              f"{smape_s:>9.2f}%  "
              f"{len(subset):>5,}")

Evaluando 4,000 imágenes...
  0/4000...
  500/4000...
  1000/4000...
  1500/4000...
  2000/4000...
  2500/4000...
  3000/4000...
  3500/4000...
RESULTADOS FINALES DE EVALUACIÓN
Imágenes evaluadas: 4,000
Sin detección: 0
Con cervicales ignoradas: 1,987 (49.7%)
Tiempo total: 4.6 min

Ángulo      MAE (°)    SMAPE (%)
--------------------------------
PT             4.90       87.37%
MT             4.67       32.87%
TL             4.36       34.45%
--------------------------------
Total          4.64       51.56%

MAE PT excluyendo ground truth = 0.0°:
  4.95°  (3,606 imágenes)

Referencia clínica: MAE < 5° = clínicamente aceptable
Referencia literatura: SMAPE ~21.71% (mejor método AASCE2019, Wang et al. 2021)

MAE MT por severidad de la escoliosis:
  Rango                       MAE      SMAPE      N
  --------------------------------------------------
  Normal (<10°)             2.79°     40.73%    586
  Leve (10–25°)             4.29°     34.97%  2,561
  Moderada (25–40°)         5.84°   

---
### Resultados obtenidos

**Evaluación sobre 4,000 imágenes del set de prueba con el modelo versión 0.1e50:**

| Ángulo | MAE (°) | SMAPE (%) | Evaluación |
|---|---|---|---|
| PT | 4.90° | 87.37%* | Dentro del umbral clínico |
| MT | 4.67° | 32.87% | Dentro del umbral clínico |
| TL | 4.36° | 34.45% | Dentro del umbral clínico |
| **Total** | **4.64°** | **51.56%*** | Clínicamente aceptable |

*El SMAPE de PT es alto porque 394 imágenes tienen PT = 0.0° en el ground truth, el denominador de la fórmula se acerca a cero produciendo valores porcentuales artificialmente elevados. El MAE es la métrica más representativa para PT.

**MAE PT excluyendo ground truth = 0.0°:** 4.95° (3,606 imágenes)

**Imágenes con vértebras cervicales ignoradas:** 1,987 (49.7%)

**Comparación con literatura:**
El mejor método del challenge AASCE-MICCAI 2019 reportó SMAPE de 21.71%. Métodos posteriores han alcanzado SMAPE de 7.28% (MMA-Net, 2023). El presente modelo, entrenado con el dataset sintético Spinal-AI2024 en hardware
de gama media, obtiene MAE total de 4.64°.

**Análisis por severidad:**

| Severidad | MAE MT | SMAPE MT | N imágenes |
|---|---|---|---|
| Normal (<10°) | 2.79° | 40.73% | 586 |
| Leve (10–25°) | 4.29° | 34.97% | 2,561 |
| Moderada (25–40°) | 5.84° | 23.73% | 479 |
| Severa (>40°) | 8.71° | 17.89% | 374 |

El MAE aumenta con la severidad lo que es un patrón esperado en modelos entrenados con datasets no balanceados donde la mayoría de imágenes corresponden a escoliosis leve. El SMAPE disminuye con la severidad porque el modelo es proporcionalmente más preciso cuando los ángulos son grandes.

---
### 11.2 Comparación de modelos entrenados
Se entrenaron y evaluaron tres versiones del modelo para identificar el mejor. El primer entrenamiento se realizó directamente con la memoria de la laptop, el segundo entrenamiento se realizó en Colab pero la memoria acabó en la época 58, el tercer entrenamiento es la continuación del último entrenamiento hecho en Colab con la intención de completar 100 épocas:

| Modelo | Hardware | Épocas | Batch | hsv_v |
|---|---|---|---|---|
| **v1** | GTX 1650 Ti (laptop) | 50 | 8 | 0.4 |
| v1e100 | Tesla T4 (Colab) | 58 | 16 | 0.5 |
| v1e100_2 | Tesla T4 (Colab) | 50 | 16 | 0.5 |

**Resultados comparativos sobre 4,000 imágenes de prueba:**

| Métrica | v1 | v1e100 | v1e100_2 |
|---|---|---|---|
| MAE PT (°) | **4.90** | 4.83 | 4.90 |
| MAE MT (°) | **4.67** | 4.73 | 4.74 |
| MAE TL (°) | **4.36** | 4.40 | 4.35 |
| **MAE Total (°)** | **4.64** | 4.65 | 4.67 |
| SMAPE MT (%) | **32.87** | 33.03 | 33.33 |

**MAE MT por severidad:**

| Rango | v1 | v1e100 | v1e100_2 |
|---|---|---|---|
| Normal (<10°) | **2.79°** | 2.76° | 2.85° |
| Leve (10–25°) | **4.29°** | 4.30° | 4.33° |
| Moderada (25–40°) | **5.84°** | 6.12° | 6.05° |
| Severa (>40°) | **8.71°** | 8.95° | 8.82° |

**✅ Modelo seleccionado: v1** (MAE total: 4.64°)

El modelo v1, entrenado en hardware local con 50 épocas, superó a los modelos entrenados en Colab. Esto sugiere que las interrupciones del entrenamiento en Colab (sesiones desconectadas) afectaron la convergencia del optimizador, a pesar de contar con hardware más potente (Tesla T4) y
mayor batch size. La diferencia entre modelos es significativa (< 0.03° en MAE total), lo que indica que la arquitectura YOLOv11n-Pose converge de forma estable con 50 épocas sobre este dataset.

Los tres modelos están por debajo del umbral clínico de 5° en MAE total.

In [60]:
#FUNCIONES DE EVALUACIÓN
import time

def smape(predichos, reales):
    valores = []
    for pred, real in zip(predichos, reales):
        denominador = (abs(pred) + abs(real)) / 2
        if denominador > 0:
            valores.append(abs(pred-real) / denominador * 100)
    return np.mean(valores) if valores else 0.0


def evaluar_modelo(ruta_modelo, nombre_modelo):
    from ultralytics import YOLO

    modelo_eval = YOLO(ruta_modelo)
    errores_pt = []
    errores_mt = []
    errores_tl = []
    resultados_det = []
    sin_deteccion = 0
    con_cervicales = 0

    imagenes = sorted([
        f for f in os.listdir(RUTA_IMG_TEST)
        if f.endswith(".jpg") and f in ground_truth
    ])

    print(f"\nEvaluando {nombre_modelo} "
          f"({len(imagenes):,} imágenes)...")
    inicio = time.time()

    for i, nombre_img in enumerate(imagenes):
        if i % 500 == 0:
            print(f"  {i}/{len(imagenes)}...")

        ruta_img = os.path.join(RUTA_IMG_TEST, nombre_img)
        res = modelo_eval(ruta_img, conf=0.3, verbose=False)
        r = res[0]

        if r.keypoints is None or len(r.keypoints.xy) < 2:
            sin_deteccion += 1
            continue

        kpts = r.keypoints.xy.cpu().numpy()
        pt, mt, tl, info = calcular_angulo_cobb_v2(kpts)
        gt = ground_truth[nombre_img]

        errores_pt.append(abs(pt - gt[0]))
        errores_mt.append(abs(mt - gt[1]))
        errores_tl.append(abs(tl - gt[2]))

        resultados_det.append({
            "pt_pred": pt,    "mt_pred": mt,    "tl_pred": tl,
            "pt_gt":   gt[0], "mt_gt":   gt[1], "tl_gt":   gt[2],
            "e_mt":    abs(mt - gt[1])
        })

        if info["n_cervicales"] > 0:
            con_cervicales += 1

    tiempo = time.time() - inicio

    #SMAPE
    smape_mt = smape(
        [r["mt_pred"] for r in resultados_det],
        [r["mt_gt"]   for r in resultados_det]
    )

    #MAE por severidad
    severidad = {}
    for nombre_r, min_a, max_a in [
        ("Normal (<10°)",      0,  10),
        ("Leve (10–25°)",     10,  25),
        ("Moderada (25–40°)", 25,  40),
        ("Severa (>40°)",     40, 999),
    ]:
        subset = [r["e_mt"] for r in resultados_det
                  if min_a <= r["mt_gt"] < max_a]
        severidad[nombre_r] = np.mean(subset) if subset else 0.0

    print(f"  ✅ Completado en {tiempo/60:.1f} min")
    print(f"  MAE total: "
          f"{np.mean(errores_pt+errores_mt+errores_tl):.2f}°")

    return {
        "nombre":         nombre_modelo,
        "mae_pt":         np.mean(errores_pt),
        "mae_mt":         np.mean(errores_mt),
        "mae_tl":         np.mean(errores_tl),
        "mae_total":      np.mean(errores_pt+errores_mt+errores_tl),
        "smape_mt":       smape_mt,
        "severidad":      severidad,
        "sin_deteccion":  sin_deteccion,
        "con_cervicales": con_cervicales,
        "tiempo_min":     tiempo/60,
        "n_evaluadas":    len(errores_pt)
    }

In [61]:
#Evalúa los tres modelos
resultados_v1 = evaluar_modelo(
    "./runs/pose/runs/cobb_v1/weights/best.pt",
    "v1 (laptop, 50 épocas)")

resultados_v1e100 = evaluar_modelo(
    "./runs/pose/runs/cobb_v1e100/weights/best.pt",
    "v1e100 (Colab, 58 épocas)")

resultados_v1e100_2 = evaluar_modelo(
    "./runs/pose/runs/cobb_v1e100_2/weights/best.pt",
    "v1e100_2 (Colab, continuación)")

#Tabla comparativa
print(f"\n{'='*65}")
print(f"COMPARACIÓN DE LOS TRES MODELOS")
print(f"{'='*65}")
print(f"{'Métrica':<20} {'v1':>12} {'v1e100':>12} {'v1e100_2':>12}")
print(f"{'-'*58}")

for nombre_m, key in [
    ("MAE PT (°)",    "mae_pt"),
    ("MAE MT (°)",    "mae_mt"),
    ("MAE TL (°)",    "mae_tl"),
    ("MAE Total (°)", "mae_total"),
    ("SMAPE MT (%)",  "smape_mt"),
]:
    v1 = resultados_v1[key]
    v2 = resultados_v1e100[key]
    v3 = resultados_v1e100_2[key]
    print(f"{nombre_m:<20} {v1:>11.2f}  {v2:>11.2f}  {v3:>11.2f}")

print(f"\nMAE MT por severidad:")
print(f"{'Rango':<22} {'v1':>8} {'v1e100':>8} {'v1e100_2':>10}")
print(f"{'-'*52}")
for nombre_r in resultados_v1["severidad"]:
    v1 = resultados_v1["severidad"][nombre_r]
    v2 = resultados_v1e100["severidad"][nombre_r]
    v3 = resultados_v1e100_2["severidad"][nombre_r]
    print(f"{nombre_r:<22} {v1:>7.2f}°  {v2:>7.2f}°  {v3:>9.2f}°")

# Identifica el mejor modelo
mejor_key = min(
    [("v1", resultados_v1),
     ("v1e100", resultados_v1e100),
     ("v1e100_2", resultados_v1e100_2)],
    key=lambda x: x[1]["mae_total"]
)
print(f"\n{'='*65}")
print(f"✅ Mejor modelo: {mejor_key[0]} "
      f"(MAE total: {mejor_key[1]['mae_total']:.2f}°)")
print(f"{'='*65}")


Evaluando v1 (laptop, 50 épocas) (4,000 imágenes)...
  0/4000...
  500/4000...
  1000/4000...
  1500/4000...
  2000/4000...
  2500/4000...
  3000/4000...
  3500/4000...
  ✅ Completado en 5.2 min
  MAE total: 4.64°

Evaluando v1e100 (Colab, 58 épocas) (4,000 imágenes)...
  0/4000...
  500/4000...
  1000/4000...
  1500/4000...
  2000/4000...
  2500/4000...
  3000/4000...
  3500/4000...
  ✅ Completado en 4.5 min
  MAE total: 4.65°

Evaluando v1e100_2 (Colab, continuación) (4,000 imágenes)...
  0/4000...
  500/4000...
  1000/4000...
  1500/4000...
  2000/4000...
  2500/4000...
  3000/4000...
  3500/4000...
  ✅ Completado en 4.4 min
  MAE total: 4.67°

COMPARACIÓN DE LOS TRES MODELOS
Métrica                        v1       v1e100     v1e100_2
----------------------------------------------------------
MAE PT (°)                  4.90         4.83         4.90
MAE MT (°)                  4.67         4.73         4.74
MAE TL (°)                  4.36         4.40         4.35
MAE Total (°)  

---
## 12. Validación con imágenes de entorno clínico real

Para evaluar la generalización del modelo a imágenes reales se utilizan 11 radiografías obtenidas con el apoyo de la **Dra. Georgina Waldo Benítez** y voluntarios con diagnóstico
de escoliosis.

**Características de las imágenes reales:**
- Formato: PNG, 3 canales (RGB)
- Resolución variable: 381–597px de ancho, 632–769px de alto
- Todas las imágenes fueron anonimizadas antes de su uso

A diferencia del set de prueba sintético, estas imágenes no tienen ground truth de ángulos medidos por médicos, por lo que la validación es **cualitativa** y se verifica visualmente que el modelo detecta las vértebras correctamente y calcula ángulos posibles clínicamente.

In [65]:
#VALIDACIÓN CON IMÁGENES CLÍNICAS REALES
RUTA_IMGS_REALES = "./data/imagenes_reales"

imagenes_reales = sorted([
    f for f in os.listdir(RUTA_IMGS_REALES)
    if f.endswith((".png", ".jpg", ".jpeg"))
    and not f.endswith("_anotada.png")  # ignora salidas previas
])

print(f"Imágenes encontradas: {len(imagenes_reales)}\n")

resultados_reales = []

for nombre_img in imagenes_reales:
    ruta_img  = os.path.join(RUTA_IMGS_REALES, nombre_img)
    img_orig  = cv2.imread(ruta_img)
    h, w      = img_orig.shape[:2]

    # Inferencia
    res = modelo(ruta_img, conf=0.3, verbose=False)
    r   = res[0]

    if r.keypoints is None or len(r.keypoints.xy) < 2:
        print(f"  Sin detecciones: {nombre_img}")
        continue

    kpts             = r.keypoints.xy.cpu().numpy()
    pt, mt, tl, info = calcular_angulo_cobb_v2(kpts)

    #Clasificación
    if mt < 10:
        clasificacion = "Normal (<10 grados)"
    elif mt < 25:
        clasificacion = "Leve (10-25 grados) - seguimiento"
    elif mt < 40:
        clasificacion = "Moderada (25-40 grados) - ortesis"
    else:
        clasificacion = "Severa (>40 grados) - cirugia"

    resultados_reales.append({
        "imagen":        nombre_img,
        "n_vertebras":   info["n_detectadas"],
        "n_cervicales":  info["n_cervicales"],
        "pt":            pt,
        "mt":            mt,
        "tl":            tl,
        "clasificacion": clasificacion,
        "advertencias":  info["advertencias"]
    })

    #Anotación de la imagen
    img_anotada = img_orig.copy()

    radio = max(2, int(min(h, w) * 0.01))

    colores_bgr = [
        (255, 0,   0  ),  # azul    → sup_izq
        (0,   0,   255),  # rojo    → sup_der
        (0,   255, 255),  # amarillo→ inf_izq
        (0,   255, 0  ),  # verde   → inf_der
    ]

    for kpts_vert in kpts:
        for i, (x, y) in enumerate(kpts_vert):
            cv2.circle(img_anotada,
                       (int(x), int(y)),
                       radio,
                       colores_bgr[i], -1)

    escala_fuente = max(0.4, w / 900)
    grosor_texto  = max(1, int(escala_fuente * 2))
    margen        = int(h * 0.03)  # 3% del alto

    lineas_texto = [
        f"Vertebras: {info['n_detectadas']}",
        f"PT: {pt:.1f}  MT: {mt:.1f}  TL: {tl:.1f}",
        clasificacion,
    ]
    if info["n_cervicales"] > 0:
        lineas_texto.append(
            f"Cervicales ignoradas: {info['n_cervicales']}"
        )

    for j, linea in enumerate(lineas_texto):
        y_texto = margen + j * int(escala_fuente * 35)

        cv2.putText(img_anotada, linea,
                    (margen + 1, y_texto + 1),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    escala_fuente, (0, 0, 0),
                    grosor_texto + 1)
        cv2.putText(img_anotada, linea,
                    (margen, y_texto),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    escala_fuente, (255, 255, 255),
                    grosor_texto)

    #Guarda imagen anotada
    nombre_salida = nombre_img.replace(".png", "_anotada.png") \
                               .replace(".jpg", "_anotada.jpg")
    cv2.imwrite(os.path.join(RUTA_IMGS_REALES, nombre_salida),
                img_anotada)

#Tabla de resultados
print(f"\n{'='*70}")
print(f"RESULTADOS - IMAGENES CLINICAS REALES")
print(f"{'='*70}")
print(f"{'Imagen':<20} {'N':>4} {'PT':>6} "
      f"{'MT':>6} {'TL':>6}  Clasificacion")
print(f"{'-'*70}")

for r in resultados_reales:
    adv = " (*)" if r["advertencias"] else ""
    print(f"{r['imagen']:<20} "
          f"{r['n_vertebras']:>4} "
          f"{r['pt']:>5.1f} "
          f"{r['mt']:>5.1f} "
          f"{r['tl']:>5.1f}  "
          f"{r['clasificacion']}{adv}")

if any(r["advertencias"] for r in resultados_reales):
    print(f"\n(*) Advertencia en alguna region")

print(f"\nImagenes anotadas guardadas en {RUTA_IMGS_REALES}")

Imágenes encontradas: 17

  Sin detecciones: LAT_R_000001.png
  Sin detecciones: LAT_R_000002.png
  Sin detecciones: LAT_R_000003.png
  Sin detecciones: LAT_R_000004.png
  Sin detecciones: LAT_R_000006.png
  Sin detecciones: LAT_R_000007.png
  Sin detecciones: LAT_R_000008.png
  Sin detecciones: LAT_R_000009.png
  Sin detecciones: LAT_R_000010.png
  Sin detecciones: LAT_R_000011.png

RESULTADOS - IMAGENES CLINICAS REALES
Imagen                  N     PT     MT     TL  Clasificacion
----------------------------------------------------------------------
AP_R_000001.png        16   1.2  14.1  12.8  Leve (10-25 grados) - seguimiento
AP_R_000002.png        14   0.0  18.3  18.2  Leve (10-25 grados) - seguimiento (*)
AP_R_000003.png        13   0.0   2.8   6.5  Normal (<10 grados) (*)
AP_R_000004.png         8   0.0   2.7  11.0  Normal (<10 grados) (*)
AP_R_000005.png        14   0.0   6.7   2.0  Normal (<10 grados) (*)
LAT_R_000005.png        2   0.0   0.0  17.3  Normal (<10 grados) (*)
LAT_

---
## 13. Conclusiones y trabajo futuro

### Conclusiones

Se desarrolló exitosamente un sistema de aprendizaje automático basado en **YOLOv11n-Pose** para la estimación automática de
los ángulos de Cobb PT, MT y TL en radiografías de columna vertebral en proyección anteroposterior.

**Resultados principales:**

| Métrica | Valor | Evaluación |
|---|---|---|
| MAE PT | 4.90° | < 5° umbral clínico |
| MAE MT | 4.67° | < 5° umbral clínico |
| MAE TL | 4.36° | < 5° umbral clínico |
| **MAE Total** | **4.64°** | Clínicamente aceptable |
| SMAPE MT | 32.87% | Comparable con literatura |

**Contribuciones:**
1. Se identificó que el 34.2% de las imágenes del dataset contienen vértebras cervicales extra y se desarrolló un algoritmo de asignación de regiones anclado desde L5 para manejarlas correctamente, reduciendo el MAE de 4.76° a 4.64°
2. Se validó el modelo con imágenes de entorno clínico real obtenidas con el apoyo de la Dra. Georgina Waldo Benítez y de algunos voluntarios que padecen de esta condición.

**Limitaciones:**
- Dataset de entrenamiento sintético y dataset no balanceado con aproximadamente 64% de imágenes siendo de escoliosis leve, 14.6% con columna vertebral normal, 12% de escoliosis moderada y 9.3% de escoliosis severa.
- Error aumenta con la severidad de la escoliosis (MAE severa: 8.71°) debido al dataset desbalanceado.
- El modelo no identifica vértebras individualmente por nombre anatómico y asume que las imágenes son radiografías toraco-lumbares completas con todas las vértebras visibles.

### Trabajo futuro

- Reentrenar con dataset balanceado para mejorar el MAE en escoliosis severa (>40°)
- Agregar rama lateral (vista LAT) para detección de hiperlordosis e hipolordosis.
- Fine-tuning con imágenes clínicas reales.
- Implementar identificación individual de vértebras por nombre anatómico.

# Referencias

- Chen, E., & al., e. (2024). CurvNet: Latent contour representation and iterative data engine for curvature angle estimation. Retrieved mayo 2025, from Arxiv.org: https://arxiv.org/abs/2411.12604
- Chen, K. S. (2024, July 15). Fully Automated Measurement of Cobb Angles in Coronal Plane Spine Radiographs. MDPI, 13(14), 4122 https://www.mdpi.com/2077-0383/13/14/4122.
- Lin, T.-Y. D. (2017). Feature pyramid networks for object detection. CVPR, https://doi.org/10.1109/CVPR.2017.106.
- Horng, M.-H., Kuok, C.-P., Fu, M.-J., & Lin, C.-J. S.-N. (2019). Cobb angle measurement of spine from X-ray images using convolutional neural network. Computational and Mathematical Methods in Medicine, 2019, 6357171; DOI 10.1155/2019/6357171.
- Jocher, G. &. (2024). Ultralytics YOLO11. Retrieved from GitHub: https://github.com/ultralytics/ultralytics
- Khanal, B. D. (2019, Oct 31). Automatic Cobb Angle Detection using Vertebr Detector and Vertebra Corners Regression. NAAMII, https://arxiv.org/pdf/1910.14202.
- Khanna, N., Caragea, M., & Kosmaidou, Z. (2021). EOS imaging of scoliosis, leg length discrepancy and alignment. Seminars in Musculoskeletal Radiology, 25(3), 374-386; DOI: 10.1055/s-0041-1730394.
- Rios, J. L., & al., e. (2025). Controlled comparative study of YOLOv8-Pose, YOLOv11-Pose, and Detectron2 for vertebrae detection and keypoint estimation. PLOS ONE, DOI: 10.1371/journal.pone.0347290.
- Ruoss, A. P., Pastorello, Y., & Denés, L. (2024). Automated Cobb Angle Measurements for Scoliosis Diagnosis and Assesment; AI Applications and Accuracy Enhancement Through Image Processing Techniques. Cureus, 16(8), e66736. DOI 10.7769/cureus.667.
- Slattery, C. V. (2018, Sep 1). Classifications in Brief: The Lenke Classification for Adolescent Idiopathic Scoliosis. Clinical Orthopaedics and Related Research, 11(2271-2276), 476 https://pmc.ncbi.nlm.nih.gov/articles/PMC6259994/.
- Shao, Z. Y. (2026). Latent contour representation and iterative data engine for curvature angle estimation. Pattern Recognition(112546), 172 https://doi.org/10.1016/j.patcog.2025.112546. Retrieved from https://doi.org/10.1016/j.patcog.2025.112546
- Shorten, C. &. (2019). A survey on image data augmentation for deep learning. Journal of Big Data, 6(1), 60 https://doi.org/10.1186/s40537-019-0197-0.
- Wang, L. e. (2021). Evaluation and comparison of accurate automated spinal curvature estimation algorithms with spinal anterior-posterior X-ray images. The AASCE2019 challenge. Medical Image Analysis(102115), 72 https://doi.org/10.1016/j.media.2021.102115.
- Wu, H. B. (2017). Automatic landmark estimation for adolescent idiopathic scoliosis assessment using boostnet. Medical Image Computing and Computer Assisted Intervention, 127-135.
- Ye, Q., Zhao, J., & Zhang, W. (2025). Automatic spinal Cobb angle measurements from X-ray images using a novel vertebra centroid radiation landmark detection network. Biomedical Signal Preprocessing and Control, 102, 107312; DOI: 10.1016/j.bspc.2025.107312.
- Yosinski, J. C. (2014). How transferable are features in deep neural networks. NeurIPS, https://arxiv.org/abs/1411.1792.